# Amazon Review Sentiment Analysis — Model Training

This notebook trains a sentiment classifier on Amazon food reviews using **Spark NLP + GloVe embeddings**.

## Pipeline Overview
1. Start Spark NLP session
2. Load preprocessed train / val / test data
3. Handle class imbalance (class weights + balanced sampling)
4. Build NLP feature pipeline: Document → Tokenize → Normalize → StopWords → GloVe → SentenceEmbeddings
5. Train 2 classifiers: Logistic Regression (weighted) and OneVsRest LinearSVC (balanced)
6. Evaluate on validation set — pick best model
7. Final evaluation on test set (accuracy, F1, per-class recall, balanced accuracy)
8. Save best model and label mapping to disk

## Cell 1 — Environment Setup & Spark NLP Session

We point the JVM to Java 17 (required by this environment), then start Spark via `sparknlp.start()`.
Spark NLP wraps PySpark and adds pretrained NLP models (like GloVe) directly into the Spark pipeline.

- `gpu=False` — MacBook Air has no discrete GPU usable by Spark NLP
- `memory="6g"` — safe ceiling given 8GB free RAM; leaves headroom for the OS and GloVe model

In [6]:
import os, sys

# ── Set ALL env vars BEFORE any pyspark / sparknlp import ─────────────────────
JAVA_HOME   = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
SPARK_HOME  = "/Users/beethoven/BigData/AmazonReview/.venv/lib/python3.11/site-packages/pyspark"
PYTHON_BIN  = "/Users/beethoven/BigData/AmazonReview/.venv/bin/python3"

os.environ["JAVA_HOME"]             = JAVA_HOME
os.environ["PATH"]                  = f"{JAVA_HOME}/bin:" + os.environ.get("PATH", "")
os.environ["SPARK_HOME"]            = SPARK_HOME   # overrides any stale /opt/spark from shell
os.environ["PYSPARK_PYTHON"]        = PYTHON_BIN   # kernel Python → worker Python
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_BIN
# ──────────────────────────────────────────────────────────────────────────────

import sparknlp
from sparknlp.base      import DocumentAssembler, EmbeddingsFinisher
from sparknlp.annotator import (
    Tokenizer        as NLPTokenizer,
    Normalizer,
    StopWordsCleaner,
    WordEmbeddingsModel,
    SentenceEmbeddings,
)

from pyspark.sql        import functions as F
from pyspark.ml         import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import (
    LogisticRegression,
    LinearSVC,
    OneVsRest,
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

# apple_silicon=True → arm64-native Spark NLP JAR, no Rosetta overhead
spark = sparknlp.start(gpu=False, apple_silicon=True, memory="6G")
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark NLP version : {sparknlp.version()}")
print(f"Spark version     : {spark.version}")
print(f"Java home         : {os.environ['JAVA_HOME']}")
print(f"Spark home        : {os.environ['SPARK_HOME']}")

26/05/06 23:46:21 WARN Utils: Your hostname, Louays-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.25 instead (on interface en0)
26/05/06 23:46:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/beethoven/BigData/AmazonReview/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/beethoven/.ivy2/cache
The jars for the packages stored in: /Users/beethoven/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp-silicon_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7d2b0b4a-319b-4fbd-9f85-89efe7ba79c9;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp-silicon_2.12;6.4.0 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in local-m2-cache
	found commons-codec#commons-codec;1.15 in local-m2-cache
	found org.apache.httpcomponents#httpclient;4.5.13 in local-m2-cache
	found org.apache.httpcomponents#httpcore;4.4.13 in local-m2-cache
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## Cell 2 — Load Preprocessed Datasets

We load the three CSV files produced by `01_eda_preprocessing.ipynb`.
Only the columns we actually need are kept to save memory.

- `clean_text` — lemmatized, stopword-free review text (our model input)
- `sentiment`  — our target label: negative / neutral / positive
- `ProductId`, `UserId`, `Id`, `Time` — kept for MongoDB storage and dashboard queries later

`repartition(8)` spreads data across 8 parallel Spark tasks — speeds up all subsequent operations.

In [ ]:
BASE = "/Users/beethoven/BigData/AmazonReview"
COLS = ["clean_text", "sentiment", "ProductId", "UserId", "Id", "Time"]

def load(path):
    return (
        spark
        .createDataFrame(pd.read_csv(path)[COLS].dropna())
        .repartition(8)
    )

train = load(f"{BASE}/data/train.csv")
val   = load(f"{BASE}/data/val.csv")
test  = load(f"{BASE}/data/test.csv")

print(f"Train : {train.count():,}")
print(f"Val   : {val.count():,}")
print(f"Test  : {test.count():,}")

## Cell 3 — Handle Class Imbalance

The dataset is heavily skewed: ~78% positive, ~14% negative, ~8% neutral.
Without correction the model learns to predict 'positive' for everything and still scores ~78% accuracy — which is useless.

We use **two complementary strategies**:

1. **Class weights** (`train_w`) — mathematically penalise the model more for misclassifying rare classes.
   Formula: `weight = total / (num_classes × class_count)`
   Used by Logistic Regression which natively supports `weightCol`.

2. **Undersampling** (`balanced_train`) — cap every class at the size of the smallest class.
   Used by LinearSVC which does NOT support `weightCol` in Spark MLlib.

In [ ]:
# --- Compute class weights ---
label_counts = train.groupBy("sentiment").count().collect()
total_count  = sum(r["count"] for r in label_counts)
num_classes  = len(label_counts)

weight_map = {
    r["sentiment"]: total_count / (num_classes * r["count"])
    for r in label_counts
}
print("Class weights:")
for sentiment, w in sorted(weight_map.items()):
    print(f"  {sentiment:10s} → {w:.4f}")

# Join weights onto training set as a new column
weights_df = spark.createDataFrame(
    [(k, float(v)) for k, v in weight_map.items()],
    ["sentiment", "classWeight"]
)
train_w = train.join(weights_df, on="sentiment", how="left")

# --- Build balanced sample (undersample majority classes) ---
min_count = min(r["count"] for r in label_counts)
print(f"\nBalanced sample: {min_count:,} rows per class")

balanced_train = (
    train.filter(F.col("sentiment") == "positive").limit(min_count)
    .union(train.filter(F.col("sentiment") == "negative").limit(min_count))
    .union(train.filter(F.col("sentiment") == "neutral" ).limit(min_count))
    .repartition(8)
)
print(f"Balanced train total: {balanced_train.count():,}")

## Cell 4 — Build the Spark NLP Feature Pipeline

This is the core upgrade over plain TF-IDF. Instead of word frequencies, we use **GloVe word embeddings** — each word is represented as a 100-dimensional vector capturing its semantic meaning.

Pipeline stages:
| Stage | What it does |
|---|---|
| `DocumentAssembler` | Converts raw string column into Spark NLP's internal Document format |
| `NLPTokenizer` | Splits text into individual word tokens |
| `Normalizer` | Lowercases and strips punctuation from tokens |
| `StopWordsCleaner` | Removes common words (the, is, at) that carry no sentiment signal |
| `GloveEmbeddings` | Maps each token to a pretrained 100-dim vector (downloads ~400MB on first run) |
| `SentenceEmbeddings` | Averages all word vectors into one vector per review |
| `EmbeddingsFinisher` | Converts Spark NLP format → standard ML DenseVector usable by classifiers |
| `StringIndexer` | Converts sentiment string → numeric label (required by all Spark ML classifiers) |

In [ ]:
document = (
    DocumentAssembler()
    .setInputCol("clean_text")
    .setOutputCol("document")
)

tokenizer = (
    NLPTokenizer()
    .setInputCols(["document"])
    .setOutputCol("token")
)

normalizer = (
    Normalizer()
    .setInputCols(["token"])
    .setOutputCol("normalized")
    .setLowercase(True)
)

stopwords_cleaner = (
    StopWordsCleaner()
    .setInputCols(["normalized"])
    .setOutputCol("clean_tokens")
    .setCaseSensitive(False)
)

# WordEmbeddingsModel.pretrained() is the spark-nlp 6.x API for GloVe 100d
glove = (
    WordEmbeddingsModel
    .pretrained("glove_100d", "en")
    .setInputCols(["document", "clean_tokens"])
    .setOutputCol("embeddings")
)

sentence_embeddings = (
    SentenceEmbeddings()
    .setInputCols(["document", "embeddings"])
    .setOutputCol("sentence_embeddings")
    .setPoolingStrategy("AVERAGE")
)

finisher = (
    EmbeddingsFinisher()
    .setInputCols(["sentence_embeddings"])
    .setOutputCols(["features"])
    .setCleanAnnotations(False)
)

indexer = StringIndexer(inputCol="sentiment", outputCol="label")

NLP_STAGES = [document, tokenizer, normalizer, stopwords_cleaner,
              glove, sentence_embeddings, finisher, indexer]

print("Pipeline stages defined. GloVe will download on first model fit.")

## Cell 5 — Train Models

We train two classifiers and compare them on the validation set using **weighted F1-score**.
We use weighted F1 (not accuracy) because the dataset is imbalanced — accuracy would be misleading.

### Classifier 1 — Logistic Regression (weighted)
- Trains on `train_w` (class-weighted full dataset)
- `family="multinomial"` — native 3-class support
- `maxIter=200`, `regParam=0.001` — more iterations, light regularisation for richer embeddings

### Classifier 2 — OneVsRest LinearSVC (balanced sample)
- LinearSVC is binary-only → OneVsRest trains 3 binary classifiers (one per class)
- Trains on `balanced_train` (undersampled) since LinearSVC has no `weightCol`
- LinearSVC often outperforms LR on high-dimensional sparse data

**Note:** GloVe downloads ~400MB on the first `pipeline.fit()` call. This is normal and only happens once.

In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

results        = {}
trained_models = {}

def train_and_eval(name, classifier, train_df):
    """Fit pipeline, evaluate on val set, store result."""
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")
    try:
        pipeline = Pipeline(stages=NLP_STAGES + [classifier])
        model    = pipeline.fit(train_df)
        f1       = evaluator.evaluate(model.transform(val))
        results[name]        = f1
        trained_models[name] = model
        print(f"✓ {name} — Val F1: {f1:.4f}")
    except Exception as exc:
        print(f"✗ {name} failed: {exc}")

# --- Classifier 1: Logistic Regression with class weights ---
lr = LogisticRegression(
    maxIter=200,
    regParam=0.001,
    elasticNetParam=0.0,
    family="multinomial",
    weightCol="classWeight"   # penalise minority class errors more
)
train_and_eval("LogisticRegression_GloVe", lr, train_w)

# --- Classifier 2: OneVsRest LinearSVC on balanced data ---
svc = LinearSVC(maxIter=100, regParam=0.01)
ovr = OneVsRest(classifier=svc)   # wraps binary SVC into multiclass
train_and_eval("OneVsRest_LinearSVC_GloVe", ovr, balanced_train)

# --- Summary ---
print("\n" + "="*50)
print("RESULTS SUMMARY (Validation F1)")
print("="*50)
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:35s}: {score:.4f}")

## Cell 6 — Select Best Model & Full Evaluation on Test Set

The best model is selected by highest validation F1.
We then run a **full diagnostic evaluation** on the held-out test set:

- **Weighted F1** — overall performance accounting for class sizes
- **Accuracy** — raw correct predictions (can be misleading with imbalance)
- **Weighted Precision / Recall** — quality of predictions per class
- **Per-class recall** — how well the model catches each sentiment individually
- **Balanced accuracy (macro recall)** — average recall across all classes equally weighted
  This is the most honest metric for imbalanced problems.

In [ ]:
# Pick best model by validation F1
best_name  = max(results, key=results.get)
best_model = trained_models[best_name]

print(f"Best model : {best_name}")
print(f"Val F1     : {results[best_name]:.4f}")
print()

# Run on test set (never seen during training or model selection)
test_preds = best_model.transform(test)

print("Test Set Metrics:")
print("-" * 35)
for metric in ["f1", "accuracy", "weightedPrecision", "weightedRecall"]:
    evaluator.setMetricName(metric)
    score = evaluator.evaluate(test_preds)
    print(f"  {metric:22s}: {score:.4f}")

# Per-class recall — exposes which sentiment the model struggles with
per_class = (
    test_preds
    .groupBy("label")
    .agg(
        F.sum(F.when(F.col("label") == F.col("prediction"), 1).otherwise(0)).alias("tp"),
        F.count("*").alias("support")
    )
    .withColumn("recall", F.col("tp") / F.col("support"))
    .orderBy("label")
)

print("\nPer-class Recall:")
per_class.select("label", "support", "recall").show(truncate=False)

balanced_acc = (
    per_class
    .select(F.avg("recall").alias("balanced_accuracy"))
    .first()["balanced_accuracy"]
)
print(f"Balanced Accuracy (macro recall): {balanced_acc:.4f}")

## Cell 7 — Save Best Model to Disk

We persist the complete trained pipeline (all NLP stages + classifier) using Spark's native format.
This saved model will be loaded by the Spark Streaming consumer to make real-time predictions.

`.write().overwrite().save()` — overwrites any previous save, safe to re-run.

In [ ]:
MODEL_PATH = "/Users/beethoven/BigData/AmazonReview/model/best_sentiment_model"

best_model.write().overwrite().save(MODEL_PATH)

print(f"Model saved  : {MODEL_PATH}")
print(f"Model type   : {best_name}")
print(f"Test F1      : see Cell 6 output")

## Cell 8 — Save Label Mapping

Spark ML classifiers output numeric predictions (0.0, 1.0, 2.0).
We need to know which number maps to which sentiment word.

`StringIndexer` sorts labels by frequency — label 0 = most frequent class (positive).
This mapping is saved as JSON and used by the streaming consumer to convert predictions back to readable labels.

In [ ]:
import json

# StringIndexer is always stage index 7 in our NLP_STAGES list
indexer_model = best_model.stages[7]
labels        = indexer_model.labels

label_mapping = {str(i): label for i, label in enumerate(labels)}

print("Label mapping (index → sentiment):")
for idx, label in label_mapping.items():
    print(f"  {idx} → {label}")

MAPPING_PATH = "/Users/beethoven/BigData/AmazonReview/model/label_mapping.json"
with open(MAPPING_PATH, "w") as f:
    json.dump(label_mapping, f, indent=2)

print(f"\nLabel mapping saved to: {MAPPING_PATH}")